# Structure learning — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Choose graph structure by balancing fit and complexity
Structure learning searches over graphs. These examples score small candidate DAGs with likelihood and BIC, show overfitting pressure, quantify edge penalties, and recover a simple dependency.

In [ ]:
# 1) score independence vs X->Y for binary data
counts=np.array([[30,10],[10,50]]); n=counts.sum(); px=counts.sum(1)/n; py=counts.sum(0)/n; pxy=counts/n
ll_ind=sum(counts[i,j]*math.log(px[i]*py[j]) for i,j in itertools.product([0,1],repeat=2)); ll_edge=sum(counts[i,j]*math.log(pxy[i,j]) for i,j in itertools.product([0,1],repeat=2))
plt.figure(figsize=(6,3)); plt.bar(['independent','X->Y'],[ll_ind,ll_edge]); plt.title('edge improves log-likelihood')
assert ll_edge>ll_ind

In [ ]:
# 2) BIC subtracts parameter cost
bic_ind=ll_ind-0.5*2*math.log(n); bic_edge=ll_edge-0.5*3*math.log(n)
plt.figure(figsize=(6,3)); plt.bar(['independent','X->Y'],[bic_ind,bic_edge]); plt.title('BIC keeps the useful edge')
assert bic_edge>bic_ind

In [ ]:
# 3) weak dependence may not pay for an edge
weak=np.array([[28,22],[22,28]]); n=weak.sum(); px=weak.sum(1)/n; py=weak.sum(0)/n; pxy=weak/n
li=sum(weak[i,j]*math.log(px[i]*py[j]) for i,j in itertools.product([0,1],repeat=2)); le=sum(weak[i,j]*math.log(pxy[i,j]) for i,j in itertools.product([0,1],repeat=2))
bi=li-0.5*2*math.log(n); be=le-0.5*3*math.log(n)
plt.figure(figsize=(6,3)); plt.bar(['no edge','edge'],[bi,be]); plt.title('penalty rejects weak edge')
assert bi>be

In [ ]:
# 4) edge penalty grows with sample size only logarithmically
ns=np.arange(20,500); pen=0.5*np.log(ns)
plt.figure(figsize=(6,3)); plt.plot(ns,pen); plt.xlabel('n'); plt.ylabel('one extra parameter penalty'); plt.title('BIC complexity cost = 0.5 log n')
assert pen[-1]>pen[0]

In [ ]:
# 5) mutual information is the likelihood gain per sample
mi=(ll_edge-ll_ind)/100
plt.figure(figsize=(6,3)); plt.bar(['MI nats'],[mi]); plt.title(f'I(X;Y)={mi:.3f}')
assert abs(mi-0.1777408838419504)<1e-12